# 10 -- Baseline vs. candidate release comparison

Paired script: `analysis/compare_releases.py`.

**Corrected, 2026-07-22 Codex review finding (fifth round, R5F17): this notebook previously
claimed BOTH the win-rate and R-expectancy differences use a two-sample bootstrap interval.
That was only ever true for R-expectancy.** The win-rate difference uses a proper
two-proportion interval (Newcombe-Wilson, `metrics.wilson_diff_confidence_interval`) --
bootstrapping raw 0/1 outcomes collapses to a degenerate interval at boundary samples (e.g.
all-win vs. all-loss groups), which is not a defensible uncertainty statement for a
proportion (see `compare_releases.py`'s own module docstring and the comment above its
`win_rate_diff` computation). Only the continuous R-expectancy difference remains a genuine
two-sample bootstrap (`two_sample_bootstrap_diff`), appropriate for a continuous statistic.
The cell below prints and asserts `summary["win_rate_diff"]["method"]` so this claim is
independently checked against the actual composed output, not just described in prose.

Per the reproducibility contract's "tiny samples cannot drive automatic changes" rule, this
never declares a release "better" automatically -- it reports a difference and its CI; the
go/no-go judgment remains a human decision.

**Uses clearly-labelled SYNTHETIC trade data for both releases.** Real-data run: PENDING.

In [ ]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.compare_releases import run

In [ ]:
def make_trades(path, exits, profits):
    rows = [
        {
            "trade_id": f"t{i}",
            "symbol": "XAUUSD",
            "is_long": "True",
            "entry_time": "2026-07-21T00:00:00Z",
            "exit_time": "2026-07-21T01:00:00Z",
            "entry_price": 100.0,
            "exit_price": e,
            "stop_price": 98.0,
            "profit": p,
        }
        for i, (e, p) in enumerate(zip(exits, profits))
    ]
    pd.DataFrame(rows).to_csv(path, index=False)


tmp_dir = Path(tempfile.mkdtemp(prefix="themba_compare_demo_"))
# Baseline: 25% win rate (5 wins / 15 losses out of 20). Candidate: 75% (15/20).
make_trades(tmp_dir / "baseline.csv", [95.0] * 15 + [105.0] * 5, [-10.0] * 15 + [10.0] * 5)
make_trades(tmp_dir / "candidate.csv", [105.0] * 15 + [95.0] * 5, [10.0] * 15 + [-10.0] * 5)

In [ ]:
# **Corrected, 2026-07-22 Codex review finding (sixth round): this cell
# previously claimed a full-YEAR period_start/period_end for a fixture
# whose trades all fall inside a single HOUR -- exactly the
# containment-vs-coverage gap compare_releases.py's own module
# docstring warns about (period_start/period_end only bound
# containment, never coverage). The claimed period below now tightly
# matches the fixture's real timestamps, and the printed
# baseline_period/candidate_period (the OBSERVED envelope, always
# returned regardless of the claim) are shown side by side with the
# claim so the distinction is visible, not just described in prose.**
summary = run(
    tmp_dir / "baseline.csv",
    tmp_dir / "candidate.csv",
    n_resamples=2000,
    seed=1,
    # period_start/period_end are now REQUIRED (Codex review finding,
    # 2026-07-22, fourth round): every trade in both datasets must fall
    # inside this caller-claimed comparison window.
    period_start="2026-07-21T00:00:00Z",
    period_end="2026-07-21T01:00:00Z",
    # broker/timeframe/modelling_mode/set_file/market_data_id/spread_note/
    # slippage_note are now REQUIRED and cross-checked for equality
    # (Codex review finding, 2026-07-22, fifth round) -- a complete
    # comparability manifest, not an optional best-effort assertion.
    baseline_broker="Deriv",
    candidate_broker="Deriv",
    baseline_timeframe="M5",
    candidate_timeframe="M5",
    baseline_modelling_mode="every_tick",
    candidate_modelling_mode="every_tick",
    baseline_set_file="default.set",
    candidate_set_file="default.set",
    baseline_market_data_id="synthetic-fixture-v1",
    candidate_market_data_id="synthetic-fixture-v1",
    baseline_spread_note="2-pip fixed spread assumed",
    candidate_spread_note="2-pip fixed spread assumed",
    baseline_slippage_note="no slippage modelled",
    candidate_slippage_note="no slippage modelled",
    # ea_version/data_source are now REQUIRED, not optional (Codex review
    # finding, 2026-07-22, sixth round) -- deliberately NOT required to be
    # equal between baseline/candidate, since a comparison's whole premise
    # is that the two releases legitimately differ.
    baseline_ea_version="v6.37",
    candidate_ea_version="v8.11",
    baseline_data_source="synthetic-fixture",
    candidate_data_source="synthetic-fixture",
    output_json=tmp_dir / "compare.json",
    repo_path=PROJECT_ROOT.parents[1],
)

print(f"claimed_comparison_period = {summary['claimed_comparison_period']}")
print(f"baseline_period           = {summary['baseline_period']} (OBSERVED entry/exit envelope)")
print(f"candidate_period          = {summary['candidate_period']} (OBSERVED entry/exit envelope)")
# The claim now tightly matches what was actually observed -- proving the
# containment check is meaningful here, not merely unfalsified by a vastly
# wider claim (the fixed bug this cell previously exhibited).
assert summary["claimed_comparison_period"] == ["2026-07-21T00:00:00Z", "2026-07-21T01:00:00Z"]
assert summary["baseline_period"][0].startswith("2026-07-21")
assert summary["candidate_period"][0].startswith("2026-07-21")

print(f"baseline_win_rate    = {summary['baseline_win_rate']:.4f}")
print(f"candidate_win_rate   = {summary['candidate_win_rate']:.4f}")
print(
    f"win_rate_diff        = {summary['win_rate_diff']['observed_diff']:.4f} "
    f"(95% CI [{summary['win_rate_diff']['ci_lower']:.4f}, {summary['win_rate_diff']['ci_upper']:.4f}])"
)
print(f"likely_significant   = {summary['win_rate_diff']['likely_significant']}")

assert abs(summary["baseline_win_rate"] - 0.25) < 1e-9
assert abs(summary["candidate_win_rate"] - 0.75) < 1e-9
assert summary["win_rate_diff"]["likely_significant"] is True

# **Added, 2026-07-22 Codex review finding (fifth round, R5F17): hand-check
# the ACTUAL mechanism each difference uses, not just that the cell ran.**
print(f"win_rate_diff method = {summary['win_rate_diff']['method']!r}")
assert summary["win_rate_diff"]["method"] == "newcombe_wilson"

# R-expectancy diff: every baseline trade has entry=100, stop=98, so
# r_multiple = (exit_price - 100) / (100 - 98) = (exit_price - 100) / 2.
# Baseline: 15 trades exit=95.0 -> r=-2.5, 5 trades exit=105.0 -> r=+2.5;
# mean = (15*(-2.5) + 5*(2.5)) / 20 = -1.25.
# Candidate: 15 trades exit=105.0 -> r=+2.5, 5 trades exit=95.0 -> r=-2.5;
# mean = (15*(2.5) + 5*(-2.5)) / 20 = +1.25.
# observed_diff = mean(candidate) - mean(baseline) = 1.25 - (-1.25) = 2.5 --
# an exact point statistic on the real data (not a resampled quantity, see
# two_sample_bootstrap_diff's own docstring), so this is asserted exactly;
# the resampled CI bounds are stochastic (though seeded) and are printed,
# not hard-asserted.
#
# **Corrected, 2026-07-22 Codex review finding (sixth round): this cell
# previously claimed "+-2.5 is a TOTAL SEPARATION between the two
# groups, so no resample can plausibly straddle 0" -- FALSE. Both groups
# are a MIX of -2.5 and +2.5 (baseline: 15 at -2.5, 5 at +2.5; candidate
# the reverse 15/5 split), not two disjoint sets, so a resample COULD in
# principle draw an extreme, same-sign-dominated sample from either
# group. The real reason likely_significant is True here is
# probabilistic, not structural: with a 75/25-majority split at n=20 per
# resample, the exact probability that one candidate resample's mean
# falls at or below one baseline resample's mean is only
# approximately 0.0005724311 (a binomial-tail calculation on the
# resampling distribution, not zero) -- small enough that
# likely_significant is True for virtually every seed in practice, but
# this is NOT the same claim as "no resample can straddle zero".**
print(
    f"expectancy_r_diff         = {summary['expectancy_r_diff']['observed_diff']:.4f} "
    f"(95% CI [{summary['expectancy_r_diff']['ci_lower']:.4f}, "
    f"{summary['expectancy_r_diff']['ci_upper']:.4f}])"
)
assert abs(summary["baseline_expectancy_r"] - (-1.25)) < 1e-9
assert abs(summary["candidate_expectancy_r"] - 1.25) < 1e-9
assert abs(summary["expectancy_r_diff"]["observed_diff"] - 2.5) < 1e-9
assert summary["expectancy_r_diff"]["likely_significant"] is True

# **Added, 2026-07-22 Codex review finding (sixth round): this notebook
# previously never inspected baseline_summary/candidate_summary/
# surface_diff at all, despite the canonical docs claiming the new
# comparison surface is hand-checked. Hand-traced against the SAME
# fixture above (all 20 baseline/candidate trades share one exit_time
# each, so each dataset collapses into ONE balance step):
# baseline net_profit = 15*(-10) + 5*(10) = -100; candidate net_profit =
# 15*(10) + 5*(-10) = +100 -> surface_diff.net_profit = 100 - (-100) = 200.
# baseline profit_factor = gross_profit(50) / gross_loss(150) = 1/3;
# candidate = gross_profit(150) / gross_loss(50) = 3.0.
# baseline's one balance step is negative (-100) -> longest_losing_
# balance_step_streak = 1; candidate's one step is positive -> streak = 0.
# candidate's balance curve [1000, 1100] never draws down -> its own
# recovery_factor is None (no drawdown to divide by), so the point diff
# is also None (never a fabricated number when either side is undefined).
baseline_summary = summary["baseline_summary"]
candidate_summary = summary["candidate_summary"]
surface_diff = summary["surface_diff"]
print(f"baseline_summary.net_profit      = {baseline_summary['net_profit']:.2f}")
print(f"candidate_summary.net_profit     = {candidate_summary['net_profit']:.2f}")
print(f"surface_diff.net_profit          = {surface_diff['net_profit']:.2f}")
print(f"baseline_summary.profit_factor   = {baseline_summary['profit_factor']:.4f}")
print(f"candidate_summary.profit_factor  = {candidate_summary['profit_factor']:.4f}")
print(f"surface_diff.profit_factor       = {surface_diff['profit_factor']:.4f}")
print(
    f"longest_losing_balance_step_streak (baseline, candidate) = "
    f"{baseline_summary['longest_losing_balance_step_streak']}, "
    f"{candidate_summary['longest_losing_balance_step_streak']}"
)
print(f"candidate_summary.recovery_factor = {candidate_summary['recovery_factor']!r}")
print(f"surface_diff.recovery_factor      = {surface_diff['recovery_factor']!r}")

assert abs(baseline_summary["net_profit"] - (-100.0)) < 1e-9
assert abs(candidate_summary["net_profit"] - 100.0) < 1e-9
assert abs(surface_diff["net_profit"] - 200.0) < 1e-9
assert abs(baseline_summary["profit_factor"] - (1.0 / 3.0)) < 1e-9
assert abs(candidate_summary["profit_factor"] - 3.0) < 1e-9
assert abs(surface_diff["profit_factor"] - (3.0 - 1.0 / 3.0)) < 1e-9
assert baseline_summary["longest_losing_balance_step_streak"] == 1
assert candidate_summary["longest_losing_balance_step_streak"] == 0
assert surface_diff["longest_losing_balance_step_streak"] == -1
assert candidate_summary["recovery_factor"] is None
assert surface_diff["recovery_factor"] is None

## Real-data run: PENDING

Requires two real trade histories (baseline release vs. a candidate release) -- neither exists yet.